# Environmental MusicGen: LoRA Fine-Tuning on ESC-50

This notebook fine-tunes MusicGen-small on environmental audio (ESC-50) using **LoRA (Low-Rank Adaptation)** adapters to generate environmental-instrumental hybrid audio.

**Project**: 10-423/623/723 Generative AI Course Project, CMU

**Before running:**
1. Enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU
2. Upload the entire `audiocraft-10623` folder to Colab (or clone from GitHub)

## ⚠️ Important Notes

- **LoRA vs Full Fine-tuning**: This notebook uses LoRA adapters, which are much more parameter-efficient than full fine-tuning. Only ~1-2% of parameters are trained.
- **Installation**: We use an optimized installation approach to avoid long build times (~10-20 min vs 1+ hour)
- **VRAM Requirements**: With LoRA, you can train on smaller GPUs. Batch size 2 works on T4 (16GB), batch size 4+ on A100 (40GB)
- **Training Time**: ~30-60 minutes per epoch on T4, ~15-30 minutes on A100 (depends on dataset size)


In [ ]:
# Mount Google Drive (Colab only - skip in Kaggle)
import os
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive mounted (Colab)")
    except ImportError:
        print("⚠️  Not in Colab - skipping Drive mount")
        print("   For Kaggle, upload your files as a dataset to /kaggle/input/")
else:
    print("Detected Kaggle environment - no Drive mount needed")
    print("Upload your audiocraft-10623 folder as a Kaggle dataset")


In [ ]:
# Navigate to your project folder (will be set properly in Step 5)
# This is just for initial exploration
import os
if os.path.exists('/kaggle/input'):
    print("Kaggle environment detected")
    print("Your files should be in /kaggle/input/audiocraft/audiocraft-10623")
    print("(They will be copied to /kaggle/working/ in Step 5)")
elif os.path.exists('/content/drive'):
    print("Colab environment detected")
    %cd /content/drive/MyDrive/10623/audiocraft-10623/
    !ls
else:
    print("Local or other environment")
    !ls


# 1. Installation

**Installation Strategy**: We install packages matching your local environment exactly:
1. PyTorch 2.1.0 (downgrade from Colab's version) - 5-10 min
2. Core dependencies (exact versions) - fast
3. xformers 0.0.22.post7 (REQUIRED, matches local) - 1-2 hours if building
4. encodec 0.1.1 (exact version) - 5-10 min  
5. audiocraft (development mode, --no-deps)
6. Remaining packages (demucs, pesq, pystoi, torchdiffeq)

**Total time: ~1.5-2.5 hours** (mostly xformers build time)
**Note**: This matches your local environment exactly to avoid compatibility issues.


In [ ]:
# Step 1: Check GPU and install correct PyTorch version
# IMPORTANT: We need PyTorch 2.1.0 to match local environment
import torch
print(f"Current PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Uninstall Kaggle's PyTorch and install matching version
print("\nInstalling PyTorch 2.1.0 to match local environment...")
print("⚠️  IMPORTANT: After this cell completes, RESTART THE KERNEL (Kernel → Restart)")
print("   Then re-run this cell to verify the version is correct.")
!pip uninstall -y torch torchvision torchaudio torchtext
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 torchtext==0.16.0 --index-url https://download.pytorch.org/whl/cu118

# Force reload torch module (may not work, kernel restart is more reliable)
import importlib
import sys
if 'torch' in sys.modules:
    del sys.modules['torch']
    if 'torchvision' in sys.modules:
        del sys.modules['torchvision']
    if 'torchaudio' in sys.modules:
        del sys.modules['torchaudio']

# Verify (may still show old version until kernel restart)
import torch
print(f"\nPyTorch version after install: {torch.__version__}")
print("⚠️  If this still shows 2.6.0, RESTART THE KERNEL and re-run this cell!")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  Warning: No GPU detected. Training will be very slow!")


In [ ]:
# Step 2: Install core dependencies matching local versions
# Install in groups to identify any failures
print("Installing core dependencies (matching local versions)...")

# Install system dependencies first (for Kaggle/Colab)
print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq libsndfile1 ffmpeg

# Group 1: Simple packages (usually have pre-built wheels)
print("Installing group 1 (simple packages)...")
!pip install -q einops==0.8.1 num2words==0.5.14 "numpy<2.0.0" sentencepiece==0.2.1 huggingface_hub==0.36.0 tqdm==4.67.1 protobuf==6.33.1 pyyaml==6.0.3

# Group 2: Audio packages (need system libs)
print("Installing group 2 (audio packages)...")
!pip install -q soundfile==0.13.1 librosa==0.11.0

# Group 3: ML/AI packages
print("Installing group 3 (ML packages)...")
!pip install -q "transformers==4.33.0" torchmetrics==1.8.2 julius==0.2.7

# Group 4: Hydra and related
print("Installing group 4 (Hydra)...")
!pip install -q "hydra-core>=1.1" hydra_colorlog==1.2.0

# Group 5: Flashy and dora (might have dependencies)
print("Installing group 5 (flashy, dora)...")
!pip install -q "flashy>=0.0.1" dora_search==0.1.12

# Group 6: spaCy (can be tricky)
print("Installing group 6 (spaCy)...")
!pip install -q "spacy==3.7.6"

# Group 7: av (PyAV - often problematic, install last)
# PyAV requires additional system libraries
print("Installing group 7 (av/PyAV)...")
print("  Installing PyAV system dependencies...")
!apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config

print("  Installing PyAV (this may take a few minutes)...")
# Try exact version first, if it fails, install latest
import subprocess
import sys

result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av==11.0.0'], 
                       capture_output=True, text=True)
if result.returncode != 0:
    print("  ⚠️  av==11.0.0 failed, trying latest version...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av'], check=False)
    print("  ✓ av installed (latest version)")
else:
    print("  ✓ av==11.0.0 installed")

print("✓ Core dependencies installed")


In [ ]:
# Step 3: Install xformers (REQUIRED, matching local version)
print("Installing xformers==0.0.22.post7 (this may take 1-2 hours if building from source)...")
print("⚠️  This is required for audiocraft. It will take time but is necessary.")
!pip install -q xformers==0.0.22.post7 --no-build-isolation
print("✓ xformers installed")


In [ ]:
# Step 4: Install encodec (matching local version)
print("Installing encodec==0.1.1 (this may take 5-10 minutes)...")
!pip install -q encodec==0.1.1
print("✓ encodec installed")


In [ ]:
# Step 5: Install audiocraft in development mode
# Use --no-deps since we've installed dependencies manually
print("Installing audiocraft in development mode...")

# For Kaggle: /kaggle/input/ is read-only, so copy to /kaggle/working/ first
# For Colab: Use the original path
import os
if os.path.exists('/kaggle/input'):
    # Kaggle environment
    print("  Detected Kaggle environment - copying to working directory...")
    import shutil
    source_dir = '/kaggle/input/audiocraft/audiocraft-10623'  # Adjust if your dataset name is different
    working_dir = '/kaggle/working/audiocraft-10623'
    
    if not os.path.exists(working_dir):
        print(f"  Copying from {source_dir} to {working_dir}...")
        shutil.copytree(source_dir, working_dir)
        print("  ✓ Files copied")
    else:
        print(f"  ✓ Files already in {working_dir}")
    
    os.chdir(working_dir)
    print(f"  Working directory: {working_dir}")
else:
    # Colab environment
    os.chdir('/content/drive/MyDrive/10623/audiocraft-10623')
    print(f"  Working directory: /content/drive/MyDrive/10623/audiocraft-10623")

!pip install -q -e . --no-deps --no-build-isolation
print("✓ audiocraft installed")


In [ ]:
# Step 6: Install remaining packages matching local versions
print("Installing remaining packages (matching local versions)...")
!pip install -q --no-build-isolation demucs==4.0.1 pesq==0.0.4 pystoi==0.4.1 torchdiffeq==0.2.5

# Try laion-clap (optional for evaluation)
import subprocess
result = subprocess.run(['pip', 'install', '-q', 'laion-clap'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ laion-clap installed")
else:
    print("⚠️  laion-clap installation failed (optional, can skip)")

print("✓ All packages installed")


In [ ]:
# Step 7: Verify installation
print("Verifying installation...")
try:
    import audiocraft
    print(f"✓ audiocraft imported (version: {getattr(audiocraft, '__version__', 'unknown')})")
except ImportError as e:
    print(f"✗ Error importing audiocraft: {e}")
    print("  Try running: !pip install -e . --no-build-isolation")

try:
    from audiocraft.models.musicgen import MusicGen
    print("✓ MusicGen imported successfully")
except ImportError as e:
    print(f"✗ Error importing MusicGen: {e}")

try:
    import encodec
    print("✓ encodec imported successfully")
except ImportError as e:
    print(f"✗ Error importing encodec: {e}")

try:
    import flashy
    print("✓ flashy imported successfully")
except ImportError as e:
    print(f"✗ Error importing flashy: {e}")

print("\n✓ Installation verification complete!")


In [ ]:
# Step 8: Setup paths (works for both Kaggle and Colab)
import os
import sys
from pathlib import Path

# Detect environment and set paths accordingly
if os.path.exists('/kaggle/working'):
    # Kaggle environment
    REPO_ROOT = '/kaggle/working/audiocraft-10623'
    PROJECT_DIR = f'{REPO_ROOT}/10623'
    print("Detected Kaggle environment")
elif os.path.exists('/content/drive'):
    # Colab environment
    REPO_ROOT = '/content/drive/MyDrive/10623/audiocraft-10623'
    PROJECT_DIR = f'{REPO_ROOT}/10623'
    print("Detected Colab environment")
else:
    # Local or other environment
    REPO_ROOT = os.getcwd()
    PROJECT_DIR = f'{REPO_ROOT}/10623'
    print("Using current working directory")

# Add to Python path
if Path(REPO_ROOT).exists():
    sys.path.insert(0, REPO_ROOT)
    if Path(PROJECT_DIR).exists():
        sys.path.insert(0, PROJECT_DIR)
    os.chdir(REPO_ROOT)
    print(f"✓ Repo found at {REPO_ROOT}")
    if Path(PROJECT_DIR).exists():
        print(f"✓ Project directory: {PROJECT_DIR}")
    else:
        print(f"⚠️  Project directory not found: {PROJECT_DIR}")
else:
    print(f"⚠️  Warning: {REPO_ROOT} not found!")
    print("Please upload/copy the audiocraft-10623 folder to the appropriate location")
    print("  - Kaggle: /kaggle/input/audiocraft/audiocraft-10623 (will be copied to /kaggle/working/)")
    print("  - Colab: /content/drive/MyDrive/10623/audiocraft-10623")


# 2. Download ESC-50 Dataset

ESC-50 is a dataset of 2,000 environmental sound recordings organized into 50 semantic classes.

**Dataset Structure:**
```
ESC-50/
  audio/
    1-100032-A-0.wav
    ...
  meta/
    esc50.csv
```

**Note**: You can also use UrbanSound8K by modifying the dataset loader, but ESC-50 is the primary dataset for this project.


In [ ]:
# Download ESC-50 dataset
# Set path based on environment
if os.path.exists('/kaggle/working'):
    # Kaggle: use working directory
    ESC50_ROOT = '/kaggle/working/audiocraft-10623/10623/ESC-50'
elif os.path.exists('/content/drive'):
    # Colab: use drive
    ESC50_ROOT = '/content/drive/MyDrive/10623/audiocraft-10623/10623/ESC-50'
else:
    # Local or other
    ESC50_ROOT = str(Path(REPO_ROOT) / '10623' / 'ESC-50')

os.environ['ESC50_ROOT'] = ESC50_ROOT

if not Path(ESC50_ROOT).exists():
    print("Downloading ESC-50 dataset...")
    !wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O /tmp/ESC-50.zip
    !unzip -q /tmp/ESC-50.zip -d /tmp/
    !mv /tmp/ESC-50-master {ESC50_ROOT}
    !rm /tmp/ESC-50.zip
    print(f"✓ ESC-50 downloaded to {ESC50_ROOT}")
else:
    print(f"✓ ESC-50 already exists at {ESC50_ROOT}")

# Verify structure
audio_dir = Path(ESC50_ROOT) / 'audio'
meta_file = Path(ESC50_ROOT) / 'meta' / 'esc50.csv'

if audio_dir.exists() and meta_file.exists():
    print(f"✓ Dataset verified: {len(list(audio_dir.glob('*.wav')))} audio files")
else:
    print(f"✗ Error: Dataset structure incorrect")
    print(f"  Expected: {audio_dir} and {meta_file}")


# 3. Test Imports

Verify that all our custom modules can be imported correctly.


In [ ]:
# Test that all imports work
print("Testing imports...")
try:
    from musicgen_lora_model import create_musicgen_lora
    print("✓ musicgen_lora_model imported")
except Exception as e:
    print(f"✗ Error importing musicgen_lora_model: {e}")

try:
    from esc50_dataset import create_esc50_dataloader
    print("✓ esc50_dataset imported")
except Exception as e:
    print(f"✗ Error importing esc50_dataset: {e}")

try:
    from lora import get_lora_parameters
    print("✓ lora imported")
except Exception as e:
    print(f"✗ Error importing lora: {e}")

try:
    from train_musicgen_esc50 import train_epoch, validate, compute_cross_entropy
    print("✓ train_musicgen_esc50 imported")
except Exception as e:
    print(f"✗ Error importing train_musicgen_esc50: {e}")

try:
    from clap_utils import CLAPEvaluator
    print("✓ clap_utils imported")
except Exception as e:
    print(f"✗ Error importing clap_utils: {e}")

print("\n✓ All imports successful!")


# 4. Configuration

Configure training parameters. Adjust these based on your GPU:

- **T4 (16GB)**: batch_size=2, works well
- **A100 (40GB)**: batch_size=4-8, faster training

**LoRA Parameters:**
- `lora_rank`: Lower rank = fewer parameters, faster training, but may limit capacity
- `lora_alpha`: Scaling factor, typically set to 2x rank
- `lora_dropout`: Regularization, 0.0-0.1 is typical


In [ ]:
# Configuration optimized for Colab
config = {
    'model_name': 'facebook/musicgen-small',
    'lora_rank': 8,  # LoRA rank (lower = fewer params, faster)
    'lora_alpha': 16.0,  # LoRA alpha (typically 2x rank)
    'lora_dropout': 0.0,  # LoRA dropout
    'dataset': {
        'root': ESC50_ROOT,
        'sample_rate': 32000,  # MusicGen uses 32kHz
        'segment_duration': None,  # Use full audio clips (or set to 10.0 for 10s segments)
        'channels': 1,  # Mono audio
    },
    'training': {
        'batch_size': 2,  # Adjust based on GPU: T4=2, A100=4-8
        'val_batch_size': 2,
        'learning_rate': 1e-4,  # Learning rate for LoRA parameters
        'weight_decay': 0.01,
        'epochs': 20,  # Number of epochs
        'num_workers': 2,  # DataLoader workers
        'max_grad_norm': 1.0,  # Gradient clipping
        'use_amp': True,  # Use mixed precision (faster, less memory)
        'scheduler': 'cosine',  # Learning rate scheduler
    },
    'eval': {
        'batch_size': 2,
        'num_workers': 2,
        'eval_num_samples': 50,  # Number of samples for evaluation
        'gen_duration': 10.0,  # Duration of generated audio in seconds
    },
    'device': 'cuda',
}

print("Configuration:")
print(f"  Model: {config['model_name']}")
print(f"  LoRA rank: {config['lora_rank']}")
print(f"  LoRA alpha: {config['lora_alpha']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Dataset: {config['dataset']['root']}")


# 5. Load Model and Create DataLoaders

Load the MusicGen model with LoRA adapters and create training/validation dataloaders.


In [ ]:
# Import training functions
from musicgen_lora_model import create_musicgen_lora
from esc50_dataset import create_esc50_dataloader
from lora import get_lora_parameters

# Load model with LoRA
print("Loading MusicGen model with LoRA...")
device = config['device'] if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = create_musicgen_lora(
    model_name=config['model_name'],
    device=device,
    lora_rank=config['lora_rank'],
    lora_alpha=config['lora_alpha'],
    lora_dropout=config['lora_dropout'],
)
print("✓ Model loaded")

# Count trainable parameters
lora_params = get_lora_parameters(model)
num_params = sum(p.numel() for p in lora_params)
print(f"✓ LoRA parameters: {num_params:,} (only these will be trained)")

# Count total parameters for reference
total_lm = sum(p.numel() for p in model.lm.parameters())
total_compression = sum(p.numel() for p in model.compression_model.parameters())
total = total_lm + total_compression
print(f"✓ Total model parameters: {total:,}")
print(f"✓ Trainable: {num_params:,} / {total:,} ({100*num_params/total:.2f}%)")


In [ ]:
# Create dataloaders
print("Creating dataloaders...")

train_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='train',
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

val_loader = create_esc50_dataloader(
    root=config['dataset']['root'],
    split='valid',
    batch_size=config['training']['val_batch_size'],
    num_workers=config['training']['num_workers'],
    segment_duration=config['dataset'].get('segment_duration'),
    sample_rate=config['dataset']['sample_rate'],
)

print("✓ Dataloaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


# 6. Setup Optimizer

Create optimizer and learning rate scheduler. Only LoRA parameters are optimized.


In [ ]:
# Create optimizer (only for LoRA parameters)
from train_musicgen_esc50 import train_epoch, validate

optimizer = torch.optim.AdamW(
    lora_params,
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
)

# Learning rate scheduler
scheduler = None
if config['training'].get('scheduler') == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['training']['epochs'],
    )

print("✓ Optimizer and scheduler created")
print(f"  Optimizer: AdamW")
print(f"  Learning rate: {config['training']['learning_rate']}")
if scheduler:
    print(f"  Scheduler: CosineAnnealingLR")


# 7. Training Loop

Train the model using LoRA adapters. Checkpoints are saved every epoch.

**Training Notes:**
- Only LoRA parameters are updated (base model is frozen)
- Training uses mixed precision (AMP) for speed
- Checkpoints save both training state and LoRA weights separately
- Best model (lowest validation loss) is saved automatically

**Resuming Training:**
To resume from a checkpoint, uncomment and modify the resume code in the cell below.


In [ ]:
# Create output directory (works for both Kaggle and Colab)
import os
if os.path.exists('/kaggle/working'):
    output_dir = Path('/kaggle/working/outputs')
elif os.path.exists('/content'):
    output_dir = Path('/content/outputs')
else:
    output_dir = Path('./outputs')

output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


In [ ]:
# Training loop
best_val_loss = float('inf')
start_epoch = 0

# Optional: Resume from checkpoint
# Uncomment and modify the path to resume training:
# resume_path = '/content/outputs/checkpoint_epoch_5_lora.pt'
# if Path(resume_path).exists():
#     print(f"Resuming from {resume_path}...")
#     model.load_lora_weights(str(resume_path))
#     checkpoint = torch.load(str(resume_path).replace('_lora.pt', '.pt'), map_location=device)
#     optimizer.load_state_dict(checkpoint['optimizer'])
#     if scheduler and 'scheduler' in checkpoint:
#         scheduler.load_state_dict(checkpoint['scheduler'])
#     start_epoch = checkpoint.get('epoch', 0) + 1
#     best_val_loss = checkpoint.get('val_metrics', {}).get('loss', float('inf'))
#     print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, config['training']['epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config['training']['epochs']}")
    print(f"{'='*60}")

    # Train
    train_metrics = train_epoch(
        model, train_loader, optimizer, device, epoch, config
    )

    # Validate
    val_metrics = validate(model, val_loader, device, config)

    # Update learning rate
    if scheduler:
        scheduler.step()

    # Log metrics
    print(f"\nTrain Loss: {train_metrics['loss']:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    if scheduler:
        print(f"Learning Rate: {scheduler.get_last_lr()[0]:.2e}")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'optimizer': optimizer.state_dict(),
    }
    if scheduler:
        checkpoint['scheduler'] = scheduler.state_dict()

    # Save model checkpoint
    checkpoint_path = output_dir / f'checkpoint_epoch_{epoch + 1}.pt'
    torch.save(checkpoint, checkpoint_path)

    # Save LoRA weights separately (much smaller, ~10-50MB)
    lora_path = output_dir / f'checkpoint_epoch_{epoch + 1}_lora.pt'
    model.save_lora_weights(str(lora_path))

    # Save best model
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_path = output_dir / 'best_checkpoint.pt'
        best_lora_path = output_dir / 'best_checkpoint_lora.pt'
        torch.save(checkpoint, best_path)
        model.save_lora_weights(str(best_lora_path))
        print(f"\n✓ Saved best model (val_loss={best_val_loss:.4f})")

print(f"\n{'='*60}")
print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Checkpoints saved to: {output_dir}")


# 8. Download Checkpoints

Download your trained LoRA weights to your local machine or Google Drive.


In [ ]:
# Download/Save checkpoints
import os
import shutil

best_lora_path = output_dir / 'best_checkpoint_lora.pt'

if os.path.exists('/content/drive'):
    # Colab: Download and save to Drive
    try:
        from google.colab import files
        if best_lora_path.exists():
            files.download(str(best_lora_path))
            print("✓ Downloaded best_checkpoint_lora.pt")
        
        # Save to Google Drive for persistence
        drive_checkpoint_dir = Path('/content/drive/MyDrive/10623/checkpoints')
        drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        if best_lora_path.exists():
            shutil.copy(best_lora_path, drive_checkpoint_dir / 'best_checkpoint_lora.pt')
            print(f"✓ Saved to Google Drive: {drive_checkpoint_dir / 'best_checkpoint_lora.pt'}")
    except ImportError:
        print("⚠️  Not in Colab - skipping download")
elif os.path.exists('/kaggle/working'):
    # Kaggle: Files are automatically saved to /kaggle/working (persists)
    if best_lora_path.exists():
        print(f"✓ Checkpoint saved to: {best_lora_path}")
        print("  In Kaggle, files in /kaggle/working are automatically saved")
        print("  You can download them from the notebook output or Kaggle's file browser")
    else:
        print("⚠️  Best checkpoint not found")
else:
    # Local or other
    if best_lora_path.exists():
        print(f"✓ Checkpoint saved to: {best_lora_path}")
    else:
        print("⚠️  Best checkpoint not found")


# 9. Evaluation

Evaluate the fine-tuned model using CLAP similarity, FAD, and KL divergence metrics.

**Metrics:**
- **CLAP Similarity**: Text-audio semantic similarity (higher is better)
- **FAD (Fréchet Audio Distance)**: Perceptual audio quality (lower is better)
- **KL Divergence**: Token distribution comparison (lower is better)


In [ ]:
# Run evaluation
# This will generate audio samples and compute metrics

import subprocess
import sys

# Load best checkpoint
best_lora_path = output_dir / 'best_checkpoint_lora.pt'
if best_lora_path.exists():
    print(f"Loading best checkpoint: {best_lora_path}")
    model.load_lora_weights(str(best_lora_path))
    print("✓ Loaded fine-tuned LoRA weights")
else:
    print("⚠️  Best checkpoint not found, using untrained model")

# Run evaluation script
# You can also run this manually: python eval_musicgen_esc50.py --checkpoint <path>
eval_script = PROJECT_DIR / 'eval_musicgen_esc50.py'
if eval_script.exists():
    print(f"Running evaluation script: {eval_script}")
    # Note: This is a simplified call - you may need to adjust arguments
    # !python {eval_script} --checkpoint {best_lora_path} --config {PROJECT_DIR}/config_esc50_lora.yaml
    print("⚠️  Uncomment the line above to run full evaluation")
    print("   Or run it manually after training completes")
else:
    print(f"⚠️  Evaluation script not found at {eval_script}")


# 10. Generation Examples

Generate environmental-instrumental hybrid audio from text prompts.

**Example Prompts:**
- "soft piano with gentle rain"
- "calming synth under forest ambience"
- "acoustic guitar with ocean waves"
- "lo-fi piano mixed with city street noise"

These prompts combine musical elements with environmental sounds, which is the core goal of this project.


In [ ]:
# Test generation with fine-tuned model
import torchaudio
import IPython.display as ipd

# Load best checkpoint if available
best_lora_path = output_dir / 'best_checkpoint_lora.pt'
if best_lora_path.exists():
    model.load_lora_weights(str(best_lora_path))
    print("✓ Loaded fine-tuned LoRA weights")
else:
    print("⚠️  Using untrained model (no checkpoint found)")

# Set generation parameters
model.set_generation_params(duration=10.0, use_sampling=True, top_k=250)

# Environmental-instrumental hybrid prompts (from proposal)
descriptions = [
    "soft piano with gentle rain",
    "calming synth under forest ambience",
    "acoustic guitar with ocean waves",
    "lo-fi piano mixed with city street noise",
    "ambient synth under city street noise",
]

print(f"Generating audio for: {descriptions}")

# Generate audio
with torch.no_grad():
    audio = model.generate(descriptions)

# Save and display generated audio
output_wav_dir = output_dir / 'wavs'
output_wav_dir.mkdir(exist_ok=True)

for i, desc in enumerate(descriptions):
    output_path = output_wav_dir / f'generated_{i:02d}_{desc.replace(" ", "_")[:30]}.wav'
    torchaudio.save(str(output_path), audio[i].cpu(), model.sample_rate)
    print(f"✓ Saved: {output_path}")
    
    # Display in notebook
    print(f"\n{desc}:")
    ipd.display(ipd.Audio(audio[i].cpu(), rate=model.sample_rate))

print("\n✓ Generation complete!")


# Notes and Troubleshooting

## Important Notes

- **Session Timeout**: Colab sessions disconnect after ~90 minutes of inactivity. Save checkpoints regularly!
- **GPU Limits**: Free tier has usage limits. Consider using Colab Pro for longer training.
- **Storage**: Colab has ~80GB disk space. ESC-50 is ~5GB, LoRA checkpoints are small (~10-50MB).
- **Resume Training**: Uncomment the resume code in cell 7 if you need to continue training.
- **Adjust Config**: Modify the config in cell 4 to change batch size, epochs, learning rate, etc.

## Troubleshooting

### Installation Issues
1. **"No module named 'audiocraft'"**: Run `!pip install -e . --no-build-isolation` in the repo root
2. **"No module named 'encodec'"**: Run `!pip install encodec` (may take 5-10 minutes)
3. **"No module named 'flashy'"**: Run `!pip install flashy>=0.0.1`
4. **xformers build fails**: You can skip xformers - it's optional but recommended for speed

### Training Issues
1. **Out of Memory (OOM)**: Reduce batch_size in config (try 1 instead of 2)
2. **NaN losses**: This can happen early in training. Check learning rate (try 5e-5 instead of 1e-4)
3. **Slow training**: Enable AMP (already enabled), or reduce segment_duration if using segments

### Model Loading Issues
1. **Can't load LoRA weights**: Make sure you're using the same LoRA rank/alpha as training
2. **Model not generating environmental sounds**: This is expected - the model needs many epochs to learn

## Differences from Full Fine-tuning

This notebook uses **LoRA** instead of full fine-tuning:
- **Pros**: Much faster training, less memory, smaller checkpoints, can train on smaller GPUs
- **Cons**: May have slightly lower capacity than full fine-tuning
- **Use Case**: Perfect for adapting pretrained models with limited compute

## Next Steps

1. **Evaluate**: Run the evaluation script to get CLAP, FAD, and KL divergence metrics
2. **Experiment**: Try different LoRA ranks, learning rates, or training durations
3. **Compare**: Compare results with baseline (untrained) model
4. **Extend**: Try mixing environmental audio with instrumental stems (as mentioned in proposal)
